In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/demand.csv")

numeric = ["holiday", "prime", "comp_shortage", "price", "discount_pct", "ad_spend", "avg_rating", "num_reviews",
          "historical_sales", "inventory_level", "shipping_time", "competitor_price_diff", "page_views", 
          "manufacturing_cost", "vendor_funding", "marketing_budget", "product_age", "time_on_platform",
          "warehouse_capacity", "distance_to_customer", "labor_level", "search_rank", "packaging_quality"]

categorical = ["quarter", "day", "category", "parent", "brand", "supplier", "fulfillment", "labor_regime", 
              "package_quality", "visibility"]

df = pd.get_dummies(df, columns=categorical, drop_first=True, dtype="int")

df["weekly_demand"] = np.log(df["weekly_demand"]+1)
df["price"] = np.log(df["price"]+1)
df["discount_pct"] = df["discount_pct"]**2
df["ad_spend"] = np.log(df["ad_spend"]+1)
df["num_reviews"] = np.log(df["num_reviews"]+1)
df["historical_sales"] = np.log(df["historical_sales"]+1)
df["inventory_level"] = np.log(df["inventory_level"]+1)
df["page_views"] = np.log(df["page_views"]+1)
df["manufacturing_cost"] = np.log(df["manufacturing_cost"]+1)
df["vendor_funding"] = np.log(df["vendor_funding"]+1)
df["marketing_budget"] = np.log(df["marketing_budget"]+1)
df["product_age"] = np.log(df["product_age"]+1)
df["time_on_platform"] = np.log(df["time_on_platform"]+1)
df["warehouse_capacity"] = np.log(df["warehouse_capacity"]+1)
df["distance_to_customer"] = np.log(df["distance_to_customer"]+1)
df["labor_level"] = df["labor_level"]**2

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

y = df["weekly_demand"]
X = df.drop(columns="weekly_demand")

full_model = LinearRegression()

full_model.fit(X, y)

full_model_preds = full_model.predict(X)

print("RMSE: ", root_mean_squared_error(y, full_model_preds))

RMSE:  0.7656583191789434


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=10)

In [7]:
train_model = LinearRegression()

train_model.fit(X_train, y_train)

test_preds = train_model.predict(X_test)

print("RMSE: ", root_mean_squared_error(y_test, test_preds))

RMSE:  0.9295528283474705


In [9]:
from sklearn.linear_model import Lasso

lasso_model = Lasso(alpha=0.2)

lasso_model.fit(X_train, y_train)

lasso_test_preds = lasso_model.predict(X_test)

print("RMSE: ", root_mean_squared_error(y_test, lasso_test_preds))

RMSE:  1.1784448639909897


In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

lasso_scale_model = Lasso(alpha=0.2)

lasso_scale_model.fit(X_train_scaled, y_train)

lasso_scale_test_preds = lasso_scale_model.predict(X_test_scaled)

print("RMSE: ", root_mean_squared_error(y_test, lasso_scale_test_preds))

RMSE:  1.1468593218026735


In [11]:
from sklearn.linear_model import LassoCV

lasso_CV_model = LassoCV(cv=10, max_iter=5000)

lasso_CV_model.fit(X_train_scaled, y_train)

lasso_CV_test_preds = lasso_CV_model.predict(X_test_scaled)

print("RMSE: ", root_mean_squared_error(y_test, lasso_CV_test_preds))

RMSE:  0.925469288744137


In [12]:
{X.columns[i]: lasso_CV_model.coef_[i] for i in range(len(X.columns))}

{'holiday': 0.16391903683368628,
 'prime': 0.041226004176818154,
 'comp_shortage': 0.28674862264440787,
 'price': -1.0066189968648636,
 'discount_pct': 0.379393815487069,
 'ad_spend': 0.0,
 'avg_rating': 0.3806649622415589,
 'num_reviews': 0.40217047956954455,
 'historical_sales': 0.1532400105858139,
 'inventory_level': 0.09782161838038352,
 'shipping_time': -0.35249012049717815,
 'competitor_price_diff': -0.21224730130697325,
 'page_views': 1.1793214179980016,
 'manufacturing_cost': -0.0,
 'vendor_funding': 0.08461606905883462,
 'marketing_budget': 0.6633758936582087,
 'product_age': -0.0,
 'time_on_platform': -0.0,
 'warehouse_capacity': 0.13202539446113806,
 'distance_to_customer': -0.02800902822245489,
 'labor_level': 0.08223056199858819,
 'search_rank': 0.48302903120155954,
 'packaging_quality': 0.0,
 'quarter_Q2': -0.005646340877347682,
 'quarter_Q3': -0.008150074985405951,
 'quarter_Q4-Holiday': -0.0,
 'day_Mon': -0.014364118899514791,
 'day_Sat': -0.01815484003184981,
 'day_Sun